# 🇵🇭 Philippine Address Mapping Pipeline

End-to-end pipeline: normalize → city-validate → fuzzy match → export.

**Input files required (place in the same directory as this notebook):**
- `raw_addresses.csv` — column: `order_deliveraddress`
- `ph_address_alias.csv` — columns: `pattern`, `replacement`
- `dictionary.json` — structured PH area dictionary
- `dim_location.csv` — columns: `barangay_code`, `barangay_name`, `city_code`, `city_name`, `province_code`, `province_name`, `region_code`, `region_name`

In [14]:
# Cell 1 — Install dependencies
# Run once; safe to re-run
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['rapidfuzz', 'openpyxl', 'pandas', 'tqdm']:
    try:
        __import__(pkg.split('[')[0])
        print(f'✅ {pkg} already installed')
    except ImportError:
        print(f'📦 Installing {pkg}...')
        install(pkg)
        print(f'✅ {pkg} installed')

print('\n🎉 All dependencies satisfied')

✅ rapidfuzz already installed
✅ openpyxl already installed
✅ pandas already installed
✅ tqdm already installed

🎉 All dependencies satisfied


In [15]:
# Cell 2 — Imports & configuration
import os, re, json, warnings, datetime
from pathlib import Path

import pandas as pd
from rapidfuzz import fuzz, process as fuzz_process
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ── File paths (edit here if files are elsewhere) ──────────────────────────
RAW_ADDR_FILE     = '../../data/sample_address.xlsx'

ALIAS_FILE        = '../../data/utils/ph_address_alias_extended_v4.csv'
DICTIONARY_FILE   = '../../data/dictionary/ph_area_dictionary_v2.json'
DIM_LOCATION_FILE = '../../data/mapping/dim_location_20260415_v3.csv'

# ── Output directories ──────────────────────────────────────────────────────
OUT_VALID   = Path('../../data/output/valid')
OUT_INVALID = Path('../../data/output/invalid')
OUT_LOGS    = Path('../../data/output/logs')
OUT_COMBINED = Path('../../data/output/combined')
OUT_REVIEW   = Path('../../data/output/review')
for d in [OUT_VALID, OUT_INVALID, OUT_LOGS, OUT_COMBINED, OUT_REVIEW]:
    d.mkdir(parents=True, exist_ok=True)

TODAY = datetime.datetime.now().strftime('%Y%m%d')
RUN_TS = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Fuzzy threshold
FUZZY_THRESHOLD = 80

print(f'✅ Config ready — run timestamp: {RUN_TS}')

✅ Config ready — run timestamp: 2026-04-17 09:57:23


In [16]:
# Cell 3 — Load all input files

# ── 3a: Raw addresses (single file OR batch files) ──────────────────────────
def _safe_read_with_encoding(filepath: str) -> pd.DataFrame:
    """Try encodings in correct order to preserve ñ, é, etc."""
    for encoding in ['utf-8-sig', 'utf-8', 'cp1252', 'iso-8859-1']:
        try:
            df = pd.read_csv(filepath, dtype=str, encoding=encoding).fillna('')
            print(f'   ✓ Read with {encoding}: {filepath}')
            return df
        except (UnicodeDecodeError, LookupError):
            continue

    df = pd.read_csv(filepath, dtype=str, encoding='cp1252', errors='replace').fillna('')
    print(f'   ⚠ Read with cp1252 + error handling: {filepath}')
    return df

def _read_any(path_obj: Path) -> pd.DataFrame:
    suffix = path_obj.suffix.lower()
    if suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path_obj, dtype=str).fillna('')
    return _safe_read_with_encoding(str(path_obj))

# Prefer batch mode when input_paths contains existing files; fallback to RAW_ADDR_FILE
batch_files = [Path(p) for p in input_paths if Path(p).exists()]

if batch_files:
    print(f'📦 Batch mode: {len(batch_files)} file(s)')
    raw_parts = []
    for p in batch_files:
        part_df = _read_any(p)
        assert 'order_deliveraddress' in part_df.columns, \
            f'Missing order_deliveraddress in {p.name}. Columns: {part_df.columns.tolist()}'
        raw_parts.append(part_df[['order_deliveraddress']])
    raw_df = pd.concat(raw_parts, ignore_index=True)
    raw_source_label = f'batches ({len(batch_files)} files)'
else:
    print(f'📄 Single-file mode: {RAW_ADDR_FILE}')
    raw_df = _read_any(Path(RAW_ADDR_FILE))
    assert 'order_deliveraddress' in raw_df.columns, \
        f'Expected column order_deliveraddress, got: {raw_df.columns.tolist()}'
    raw_df = raw_df[['order_deliveraddress']]
    raw_source_label = RAW_ADDR_FILE

raw_addresses = raw_df['order_deliveraddress'].astype(str).fillna('').tolist()
total_raw = len(raw_addresses)
print(f'📂 Raw addresses loaded   : {total_raw:,}')
print(f'   Source                : {raw_source_label}')

# ── 3b: Alias substitution rules (encoding-safe) ────────────────────────────
alias_df = _safe_read_with_encoding(ALIAS_FILE)
assert 'pattern' in alias_df.columns and 'replacement' in alias_df.columns, \
    'Alias file must have pattern and replacement columns'
alias_rules = list(zip(alias_df['pattern'].str.strip(), alias_df['replacement'].str.strip()))
print(f'📂 Alias rules loaded      : {len(alias_rules)}')

# ── 3c: Dictionary ─────────────────────────────────────────────────────────
try:
    with open(DICTIONARY_FILE, 'r', encoding='utf-8-sig') as f:
        dictionary = json.load(f)
except (UnicodeDecodeError, json.JSONDecodeError):
    with open(DICTIONARY_FILE, 'r', encoding='utf-8') as f:
        dictionary = json.load(f)

norm_settings    = dictionary.get('normalization', {})
global_rules     = dictionary.get('global_rules', {})
anchor_tokens    = global_rules.get('anchor_tokens', [])
dict_cities      = dictionary.get('cities', {})          # {CITY_NAME: {aliases:[...], ...}}
province_aliases = dictionary.get('province_aliases', {}) # {PROVINCE: [alias,...]}
all_cities_list  = dictionary.get('all_cities', [])

# Build flat alias → canonical maps
city_alias_map = {}  # alias_upper -> canonical city name
for canonical, meta in dict_cities.items():
    city_alias_map[canonical.upper()] = canonical
    for alias in meta.get('aliases', []):
        city_alias_map[alias.upper()] = canonical

province_alias_map = {}  # alias_upper -> canonical province name
for canonical, aliases in province_aliases.items():
    province_alias_map[canonical.upper()] = canonical
    for alias in aliases:
        province_alias_map[alias.upper()] = canonical

print(f'📂 Dictionary loaded       : {len(dict_cities)} city entries, '
      f'{len(province_aliases)} province alias groups')

# ── 3d: dim_location (encoding-safe) ──────────────────────────────────────
dim_df = _safe_read_with_encoding(DIM_LOCATION_FILE)

# Normalise column names (accept both naming conventions)
dim_df.columns = [c.strip().lower() for c in dim_df.columns]

REQUIRED_DIM_COLS = ['barangay_code', 'barangay_name', 'city_code',
                     'city_name', 'province_code', 'province_name', 'region_code', 'region_name']
missing_cols = [c for c in REQUIRED_DIM_COLS if c not in dim_df.columns]
assert not missing_cols, f'dim_location missing columns: {missing_cols}'

# Uppercase reference columns for matching
dim_df['barangay_name_up'] = dim_df['barangay_name'].str.upper().str.strip()
dim_df['city_name_up']     = dim_df['city_name'].str.upper().str.strip()
dim_df['province_name_up'] = dim_df['province_name'].str.upper().str.strip()
dim_df['region_name_up']   = dim_df['region_name'].str.upper().str.strip()


print(f'📂 dim_location loaded     : {len(dim_df):,} rows')
print(f'\n✅ All input files loaded successfully')

📦 Batch mode: 1 file(s)
📂 Raw addresses loaded   : 10,000
   Source                : batches (1 files)
   ✓ Read with cp1252: ../../data/utils/ph_address_alias_extended_v4.csv
📂 Alias rules loaded      : 523
📂 Dictionary loaded       : 165 city entries, 18 province alias groups
   ✓ Read with cp1252: ../../data/mapping/dim_location_20260415_v3.csv
📂 dim_location loaded     : 42,011 rows

✅ All input files loaded successfully


In [17]:
# Cell 4 — Normalization helpers

def apply_alias_rules(text: str, rules: list) -> str:
    """Apply alias substitutions from pattern -> replacement only once."""
    upper = str(text).upper()

    # Longer patterns first so "CITY OF PUERTO PRINCESA" wins over "PUERTO PRINCESA"
    for pattern, replacement in sorted(rules, key=lambda x: -len((x[0] or '').strip())):
        pattern = (pattern or '').strip()
        replacement = (replacement or '').strip().upper()
        if not pattern or not replacement:
            continue

        escaped = re.escape(pattern.upper())
        upper = re.sub(rf'(?<![\w]){escaped}(?![\w])', replacement, upper, flags=re.IGNORECASE)

    # Collapse accidental duplicate canonical city prefixes
    upper = re.sub(r'\b(CITY OF)(?:\s+\1)+\b', r'\1', upper, flags=re.IGNORECASE)
    upper = re.sub(r'\s+', ' ', upper).strip()
    return upper


def apply_dictionary_normalization(text: str) -> str:
    """Apply dictionary-based normalization: city aliases, province aliases, anchor tokens."""
    upper = text.upper()

    # Replace province aliases with canonical names
    for alias_up, canonical in province_alias_map.items():
        escaped = re.escape(alias_up)
        upper = re.sub(rf'(?<![\w])' + escaped + r'(?![\w])', canonical, upper)

    # Replace city aliases with canonical names
    for alias_up, canonical in city_alias_map.items():
        escaped = re.escape(alias_up)
        upper = re.sub(rf'(?<![\w])' + escaped + r'(?![\w])', canonical, upper)
    
    # Province aliases first (longest aliases first)
    for alias_up, canonical in sorted(province_alias_map.items(), key=lambda x: -len(x[0])):
        upper = re.sub(rf'\b{re.escape(alias_up)}\b', canonical, upper)
    
    # City aliases next (longest first), but do NOT remap if already prefixed by CITY OF
    for alias_up, canonical in sorted(city_alias_map.items(), key=lambda x: -len(x[0])):
        upper = re.sub(
            rf'(?<!CITY OF )\b{re.escape(alias_up)}\b',
            canonical,
            upper
        )

    # Safety cleanup for any repeated CITY OF
    while 'CITY OF CITY OF' in upper:
        upper = upper.replace('CITY OF CITY OF', 'CITY OF')

    # Safety cleanup for CITY OF patterns
    upper = re.sub(r'\b(CITY OF)(?:\s+\1)+\b', r'\1', upper, flags=re.IGNORECASE)
    upper = re.sub(r'\bCITY OF\s+([A-Z][A-Z\s\-.]*?)\s+CITY\b', r'CITY OF \1', upper, flags=re.IGNORECASE)
    upper = re.sub(r'\s+', ' ', upper).strip()

    return upper


def normalize_address(raw: str) -> str:
    """
    Full normalization pipeline (3 steps):
      Step 1 — alias CSV substitutions
      Step 2 — dictionary rules (global_rules, cities, province_aliases)
      Step 3 — Title Case
    """
    text = str(raw).strip()

    # Step 1: Alias CSV substitutions
    text = apply_alias_rules(raw, alias_rules)

    # Step 2: Dictionary normalization i 
    text = apply_dictionary_normalization(text)

    # Clean up extra whitespace / punctuation
    text = re.sub(r'[^\w\s,./\-]', ' ', text)  # strip odd chars
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\b(CITY OF)(?:\s+\1)+\b', r'\1', text, flags=re.IGNORECASE)
    text = re.sub(r'\bCITY OF\s+([A-Z][A-Z\s\-.]*?)\s+CITY\b', r'CITY OF \1', text, flags=re.IGNORECASE)

    # Step 3: Title Case
    text = text.title()

    return text


print('✅ Normalization helpers defined')

✅ Normalization helpers defined


In [18]:
# Cell 5 — PART 1: Normalize addresses & export normalized_addresses.csv

normalized_addresses = [normalize_address(addr) for addr in tqdm(raw_addresses, total=total_raw, desc='Normalize addresses', unit='addr')]
total_normalized = len(normalized_addresses)

norm_df = pd.DataFrame({'order_deliveraddress': normalized_addresses})
norm_df.to_csv('normalized_addresses.csv', index=False)
print(f'✅ Normalized addresses saved → normalized_addresses.csv')
print(f'   Total normalized: {total_normalized:,}')
norm_df.head(5)

Normalize addresses:   0%|          | 0/10000 [00:00<?, ?addr/s]

KeyboardInterrupt: 

In [ ]:
# Cell 6 — RTL city-scan helper (used in Part 1 filter AND Part 2 detection)

# All unique city names from dim_location (uppercase) — used for fuzzy matching
dim_city_names_up = dim_df['city_name_up'].unique().tolist()

def infer_explicit_province(address: str) -> str | None:
    up = address.upper()
    if re.search(r'\bMANILA\b', up):
        return 'METRO MANILA'
    if re.search(r'\bNCR\b', up):
        return 'METRO MANILA'
    return None

def rtl_city_scan(address: str, dim_source: pd.DataFrame, threshold: int = 90, allowed_province: str | None = None):
    """
    Scan address right-to-left, word by word, building progressively longer
    phrases and fuzzy-matching against city_choices.

    Returns (best_city_match: str | None, best_score: float)
    """
    words = address.upper().split()

    # Constrain city candidates when province is explicit
    scope = dim_source
    if allowed_province:
        scoped = dim_source[dim_source['province_name_up'] == allowed_province]
        if not scoped.empty:
            scope = scoped

    city_choices = scope['city_name_up'].dropna().unique().tolist()
    city_set = set(city_choices)

    # If province is explicit and appears at the end, remove it first
    if allowed_province:
        prov_toks = allowed_province.split()
        if len(words) >= len(prov_toks) and words[-len(prov_toks):] == prov_toks:
            words = words[:-len(prov_toks)]

    # Build suffix phrases; score exact matches by specificity instead of early return
    exact_candidates = []
    for start_idx in range(len(words) - 1, -1, -1):
        phrase = ' '.join(words[start_idx:])
        if phrase in city_set:
            specificity = len(phrase.split()) + (3 if phrase.startswith('CITY OF ') else 0)
            exact_candidates.append((specificity, phrase))

    if exact_candidates:
        exact_candidates.sort(reverse=True)
        return exact_candidates[0][1], 100.0

    # 2) Fuzzy fallback
    best_match = None
    best_score = 0.0
    
    for start_idx in range(len(words) - 1, -1, -1):
        phrase = ' '.join(words[start_idx:])
        result = fuzz_process.extractOne(
            phrase, city_choices,
            scorer=fuzz.token_set_ratio,
            score_cutoff=threshold
        )
        if result is not None:
            matched_city, score, _ = result
            adjusted = score + min(len(matched_city.split()) * 1.5, 5)
            if adjusted > best_score:
                best_score = adjusted
                best_match = matched_city

    return best_match, best_score


print('✅ RTL city-scan helper defined')
print(f'   City reference pool: {len(dim_city_names_up):,} unique cities from dim_location')

✅ RTL city-scan helper defined
   City reference pool: 1,408 unique cities from dim_location


In [ ]:
# Cell 7 — PART 1: City-check via RTL fuzzy scan → drop no-city addresses

city_validated      = []   # (normalized_address, matched_city_up)
dropped_no_city     = []

print('🔍 Running RTL city scan on normalized addresses...')

for i, addr in enumerate(tqdm(normalized_addresses, total=total_normalized, desc='City validation', unit='addr')):
    if i % 500 == 0 and i > 0:
        print(f'   Processed {i:,} / {total_normalized:,}...')

    explicit_prov = infer_explicit_province(addr)
    matched_city, score = rtl_city_scan(addr, dim_df, FUZZY_THRESHOLD, explicit_prov)
    if matched_city:
        city_validated.append((addr, matched_city))
    else:
        dropped_no_city.append(addr)

total_dropped    = len(dropped_no_city)
total_passed_p2  = len(city_validated)

print(f'\n📊 City-validation results:')
print(f'   Total normalized     : {total_normalized:,}')
print(f'   Passed city check    : {total_passed_p2:,}')
print(f'   Dropped (no city)    : {total_dropped:,}')

# Filter dim_location to only cities present in validated set
validated_city_names_up = set(city for _, city in city_validated)
filtered_dim_df = dim_df[dim_df['city_name_up'].isin(validated_city_names_up)].copy()
filtered_dim_df.reset_index(drop=True, inplace=True)

print(f'\n🗂️  Filtered dim_location  : {len(filtered_dim_df):,} rows '
      f'({filtered_dim_df["city_name_up"].nunique()} unique cities)')

🔍 Running RTL city scan on normalized addresses...


City validation:   0%|          | 0/122888 [00:00<?, ?addr/s]

   Processed 500 / 122,888...
   Processed 1,000 / 122,888...
   Processed 1,500 / 122,888...
   Processed 2,000 / 122,888...
   Processed 2,500 / 122,888...
   Processed 3,000 / 122,888...
   Processed 3,500 / 122,888...
   Processed 4,000 / 122,888...
   Processed 4,500 / 122,888...
   Processed 5,000 / 122,888...
   Processed 5,500 / 122,888...
   Processed 6,000 / 122,888...
   Processed 6,500 / 122,888...
   Processed 7,000 / 122,888...
   Processed 7,500 / 122,888...
   Processed 8,000 / 122,888...
   Processed 8,500 / 122,888...
   Processed 9,000 / 122,888...
   Processed 9,500 / 122,888...
   Processed 10,000 / 122,888...
   Processed 10,500 / 122,888...
   Processed 11,000 / 122,888...
   Processed 11,500 / 122,888...
   Processed 12,000 / 122,888...
   Processed 12,500 / 122,888...
   Processed 13,000 / 122,888...
   Processed 13,500 / 122,888...
   Processed 14,000 / 122,888...
   Processed 14,500 / 122,888...
   Processed 15,000 / 122,888...
   Processed 15,500 / 122,888..

In [ ]:
# Cell 8 — PART 2: Fuzzy matching helpers

def _score_suffix_match(address: str, choices: list[str]) -> tuple[str | None, float]:
    """Return the best exact-or-fuzzy suffix match and its score."""
    if not choices:
        return None, 0.0

    words = address.upper().split()
    choice_set = set(choices)
    exact_candidates = []
    best_match = None
    best_score = 0.0

    for start_idx in range(len(words) - 1, -1, -1):
        phrase = ' '.join(words[start_idx:])
        if phrase in choice_set:
            specificity = len(phrase.split()) + (3 if phrase.startswith('CITY OF ') else 0)
            exact_candidates.append((specificity, phrase))

        result = fuzz_process.extractOne(
            phrase, choices,
            scorer=fuzz.token_set_ratio,
            score_cutoff=0
        )
        if result is not None:
            matched_value, score, _ = result
            adjusted = score + min(len(matched_value.split()) * 1.5, 5)
            if adjusted > best_score:
                best_score = adjusted
                best_match = matched_value

    if exact_candidates:
        exact_candidates.sort(reverse=True)
        return exact_candidates[0][1], 100.0

    return best_match, round(float(best_score), 3)


def fuzzy_match_address(address: str, addr_city_up: str,
                        dim_filtered: pd.DataFrame,
                        threshold: int = 90) -> dict:
    """
    Fuzzy match a single address against its city-scoped dim_location slice.

    Fallback order:
      Level 1 → barangay_name (≥90%)
      Level 2 → city_name     (≥90%)
      Level 3 → province_name (≥90%)

    Returns a result dict with match metadata.
    """
    addr_up = address.upper()

    # Narrow candidate set to matched city
    city_slice = dim_filtered[dim_filtered['city_name_up'] == addr_city_up]
    if city_slice.empty:
        # Fallback: use all filtered dim
        city_slice = dim_filtered

    region_choices = dim_filtered['region_name_up'].dropna().unique().tolist()

    # ── Level 1: barangay ──────────────────────────────────────────────────
    brgy_choices = city_slice['barangay_name_up'].tolist()
    matched_brgy_up, brgy_score = _score_suffix_match(addr_up, brgy_choices)
    brgy_row = city_slice[city_slice['barangay_name_up'] == matched_brgy_up].iloc[0] if matched_brgy_up and matched_brgy_up in set(brgy_choices) else None

    # ── Level 2: city ──────────────────────────────────────────────────────
    city_choices = city_slice['city_name_up'].unique().tolist()
    matched_city_up, city_score = _score_suffix_match(addr_up, city_choices)
    city_row = city_slice[city_slice['city_name_up'] == matched_city_up].iloc[0] if matched_city_up and matched_city_up in set(city_choices) else None

    # ── Level 3: province ──────────────────────────────────────────────────
    prov_choices = city_slice['province_name_up'].unique().tolist()
    matched_prov_up, prov_score = _score_suffix_match(addr_up, prov_choices)
    prov_row = city_slice[city_slice['province_name_up'] == matched_prov_up].iloc[0] if matched_prov_up and matched_prov_up in set(prov_choices) else None

    # ── Region confidence (reporting only) ─────────────────────────────────
    region_score = _score_suffix_match(addr_up, region_choices)[1]

    # ── Determine the selected administrative level ───────────────────────
    if brgy_row is not None and brgy_score >= threshold:
        row = brgy_row
        level = 1
        selected_score = brgy_score
        barangay_code = row['barangay_code']
        barangay_name = row['barangay_name']
        city_code = row['city_code']
        city_name = row['city_name']
        province_code = row['province_code']
        province_name = row['province_name']
        region_code = row['region_code']
        region_name = row['region_name']
    elif city_row is not None and city_score >= threshold:
        row = city_row
        level = 2
        selected_score = city_score
        barangay_code = ''
        barangay_name = ''
        city_code = row['city_code']
        city_name = row['city_name']
        province_code = row['province_code']
        province_name = row['province_name']
        region_code = row['region_code']
        region_name = row['region_name']
    elif prov_row is not None and prov_score >= threshold:
        row = prov_row
        level = 3
        selected_score = prov_score
        barangay_code = ''
        barangay_name = ''
        city_code = row['city_code']
        city_name = row['city_name']
        province_code = row['province_code']
        province_name = row['province_name']
        region_code = row['region_code']
        region_name = row['region_name']
    else:
        level = 0
        selected_score = 0.0
        barangay_code = ''
        barangay_name = ''
        city_code = ''
        city_name = ''
        province_code = ''
        province_name = ''
        region_code = ''
        region_name = ''

    overall_confidence = round(max(brgy_score, city_score, prov_score, region_score), 3)

    if level == 0:
        return {
            'level': 0, 'score': 0.0,
            'barangay_code': '', 'barangay_name': '',
            'city_code': '', 'city_name': '',
            'province_code': '', 'province_name': '',
            'region_code': '', 'region_name': '',
            'barangay_confidence': round(brgy_score, 3),
            'city_confidence': round(city_score, 3),
            'province_confidence': round(prov_score, 3),
            'region_confidence': round(region_score, 3),
            'overall_confidence': overall_confidence,
        }

    return {
        'level': level,
        'score': round(selected_score, 3),
        'barangay_code': barangay_code,
        'barangay_name': barangay_name,
        'city_code': city_code,
        'city_name': city_name,
        'province_code': province_code,
        'province_name': province_name,
        'region_code': region_code,
        'region_name': region_name,
        'barangay_confidence': round(brgy_score, 3),
        'city_confidence': round(city_score, 3),
        'province_confidence': round(prov_score, 3),
        'region_confidence': round(region_score, 3),
        'overall_confidence': overall_confidence,
    }


print('✅ Fuzzy match function defined')

✅ Fuzzy match function defined


In [ ]:
# Cell 9 — PART 2: Run fuzzy matching on all city-validated addresses

valid_rows   = []
invalid_rows = []
combined_rows = []

level_counts = {1: 0, 2: 0, 3: 0}

print(f'🔍 Fuzzy matching {total_passed_p2:,} addresses...')

for i, (addr, matched_city_up) in enumerate(tqdm(city_validated, total=total_passed_p2, desc='Fuzzy match', unit='addr')):
    if i % 500 == 0 and i > 0:
        print(f'   Matched {i:,} / {total_passed_p2:,}...')

    explicit_prov = infer_explicit_province(addr)
    dim_scope = filtered_dim_df
    if explicit_prov:
        scoped = filtered_dim_df[filtered_dim_df['province_name_up'] == explicit_prov]
        if not scoped.empty:
            dim_scope = scoped
    result = fuzzy_match_address(
        addr, matched_city_up, dim_scope, FUZZY_THRESHOLD
    )

    row = {
        'order_deliveraddress': addr,
        'barangay_code'       : result['barangay_code'],
        'barangay_name'       : result['barangay_name'],
        'city_code'           : result['city_code'],
        'city_name'           : result['city_name'],
        'province_code'       : result['province_code'],
        'province_name'       : result['province_name'],
        'region_code'         : result['region_code'],
        'region_name'         : result['region_name'],
    }

    combined_rows.append({
        'order_deliveraddress': addr,
        'barangay_code'       : result['barangay_code'],
        'barangay_name'       : result['barangay_name'],
        'city_code'           : result['city_code'],
        'city_name'           : result['city_name'],
        'province_code'       : result['province_code'],
        'province_name'       : result['province_name'],
        'region_code'         : result['region_code'],
        'region_name'         : result['region_name'],
        'barangay_confidence' : result['barangay_confidence'],
        'city_confidence'     : result['city_confidence'],
        'province_confidence' : result['province_confidence'],
        'region_confidence'   : result['region_confidence'],
        'overall_confidence'  : result['overall_confidence'],
        'match_level'         : result['level'],
        'match_status'        : 'valid' if result['level'] > 0 else 'invalid',
    })

    if result['level'] > 0:
        valid_rows.append(row)
        level_counts[result['level']] += 1
    else:
        invalid_rows.append(row)

total_valid   = len(valid_rows)
total_invalid = len(invalid_rows)

print(f'\n📊 Matching complete:')
print(f'   ✅ Valid   : {total_valid:,}')
print(f'     Level 1 (barangay)  : {level_counts[1]:,}')
print(f'     Level 2 (city)      : {level_counts[2]:,}')
print(f'     Level 3 (province)  : {level_counts[3]:,}')
print(f'   ❌ Invalid : {total_invalid:,}')

🔍 Fuzzy matching 57,541 addresses...


Fuzzy match:   0%|          | 0/57541 [00:00<?, ?addr/s]

   Matched 500 / 57,541...
   Matched 1,000 / 57,541...
   Matched 1,500 / 57,541...
   Matched 2,000 / 57,541...
   Matched 2,500 / 57,541...
   Matched 3,000 / 57,541...
   Matched 3,500 / 57,541...
   Matched 4,000 / 57,541...
   Matched 4,500 / 57,541...
   Matched 5,000 / 57,541...
   Matched 5,500 / 57,541...
   Matched 6,000 / 57,541...
   Matched 6,500 / 57,541...
   Matched 7,000 / 57,541...
   Matched 7,500 / 57,541...
   Matched 8,000 / 57,541...
   Matched 8,500 / 57,541...
   Matched 9,000 / 57,541...
   Matched 9,500 / 57,541...
   Matched 10,000 / 57,541...
   Matched 10,500 / 57,541...
   Matched 11,000 / 57,541...
   Matched 11,500 / 57,541...
   Matched 12,000 / 57,541...
   Matched 12,500 / 57,541...
   Matched 13,000 / 57,541...
   Matched 13,500 / 57,541...
   Matched 14,000 / 57,541...
   Matched 14,500 / 57,541...
   Matched 15,000 / 57,541...
   Matched 15,500 / 57,541...
   Matched 16,000 / 57,541...
   Matched 16,500 / 57,541...
   Matched 17,000 / 57,541...
 

In [ ]:
# Cell 10 — Export valid & invalid Excel files

OUTPUT_COLS = [
    'order_deliveraddress', 'barangay_code', 'barangay_name',
    'city_code', 'city_name', 'province_code', 'province_name', 'region_code', 'region_name'
]
COMBINED_COLS = [
    'order_deliveraddress', 'barangay_code', 'barangay_name',
    'city_code', 'city_name', 'province_code', 'province_name', 'region_code', 'region_name',
    'barangay_confidence', 'city_confidence', 'province_confidence', 'region_confidence',
    'overall_confidence', 'match_level', 'match_status'
]

# ── Valid ──────────────────────────────────────────────────────────────────
valid_path = OUT_VALID / f'ps_valid_addresses_(part1){TODAY}.xlsx'
valid_out_df = pd.DataFrame(valid_rows, columns=OUTPUT_COLS)
valid_out_df.to_excel(valid_path, index=False)
print(f'✅ Valid addresses saved   → {valid_path}  ({total_valid:,} rows)')

# ── Combined ─────────────────────────────────────────────────────────────
combined_path = OUT_COMBINED / f'ps_combined_addresses_(part1){TODAY}.xlsx'
combined_out_df = pd.DataFrame(combined_rows, columns=COMBINED_COLS)
score_cols = ['barangay_confidence', 'city_confidence', 'province_confidence', 'region_confidence', 'overall_confidence']
combined_out_df[score_cols] = combined_out_df[score_cols].apply(pd.to_numeric, errors='coerce').fillna(0).round(3)
combined_out_df.to_excel(combined_path, index=False)
print(f'✅ Combined addresses saved → {combined_path}  ({len(combined_out_df):,} rows)')

# ── Invalid ────────────────────────────────────────────────────────────────
invalid_path = OUT_INVALID / f'ps_invalid_addresses_(part1){TODAY}.xlsx'
invalid_out_df = pd.DataFrame(invalid_rows, columns=OUTPUT_COLS)
invalid_out_df.to_excel(invalid_path, index=False)
print(f'✅ Invalid addresses saved → {invalid_path}  ({total_invalid:,} rows)')

# Preview
print('\n--- Valid preview (first 5) ---')
display(valid_out_df.head())
print('--- Invalid preview (first 5) ---')
display(invalid_out_df.head())

✅ Valid addresses saved   → ..\..\data\output\valid\ps_valid_addresses_(part1)20260416.xlsx  (57,541 rows)
✅ Combined addresses saved → ..\..\data\output\combined\ps_combined_addresses_(part1)20260416.xlsx  (57,541 rows)
✅ Invalid addresses saved → ..\..\data\output\invalid\ps_invalid_addresses_(part1)20260416.xlsx  (0 rows)

--- Valid preview (first 5) ---


,order_deliveraddress,barangay_code,barangay_name,city_code,city_name,province_code,province_name,region_code,region_name
0,Santo Domingo Street,9,Santo Domingo Pob.,16,Santo Domingo,5,Albay,5,Region V (Bicol Region)
1,Tagumpay Sindalan City Of San Fernando,36,Sindalan,16,San Fernando,54,Pampanga,3,Region III (Central Luzon)
2,"Block 16,Lot 4 Quiotan Street,Phil Am Life Vil...",3,Carmen,6,Carmen,68,Surigao del Sur,16,Region XIII (Caraga)
3,Cainta Rizal,6,Rizal,8,Rizal,51,Occidental Mindoro,17,MIMAROPA Region
4,2079 A Elias Street Santa Cruz,23,Santisima Cruz,26,Santa Cruz,34,Laguna,4,Region IV-A (CALABARZON)


--- Invalid preview (first 5) ---


,order_deliveraddress,barangay_code,barangay_name,city_code,city_name,province_code,province_name,region_code,region_name


In [ ]:
# Cell 11 — Isolate addresses below 70.000 confidence

low_confidence_threshold = 70.000
low_confidence_out_df = combined_out_df[combined_out_df['overall_confidence'] < low_confidence_threshold].copy()
low_confidence_out_df = low_confidence_out_df.sort_values(
    ['overall_confidence', 'order_deliveraddress'],
    ascending=[True, True]
)

low_confidence_path = OUT_REVIEW / f'low_confidence_addresses_{TODAY}.xlsx'
low_confidence_out_df.to_excel(low_confidence_path, index=False)

print(f'⚠️ Low-confidence addresses saved → {low_confidence_path}  ({len(low_confidence_out_df):,} rows)')
print(f'   Threshold: {low_confidence_threshold:.3f}')
print('')
display(low_confidence_out_df.head(20))

⚠️ Low-confidence addresses saved → ..\..\data\output\review\low_confidence_addresses_20260416.xlsx  (0 rows)
   Threshold: 70.000



,order_deliveraddress,barangay_code,barangay_name,city_code,city_name,province_code,province_name,region_code,region_name,barangay_confidence,city_confidence,province_confidence,region_confidence,overall_confidence,match_level,match_status


In [ ]:
# Cell 12 — Generate pipeline log file

log_path = OUT_LOGS / f'pipeline_log_{TODAY}.txt'

log_lines = [
    '=' * 60,
    '   PHILIPPINE ADDRESS MAPPING PIPELINE — RUN LOG',
    '=' * 60,
    f'Run Timestamp             : {RUN_TS}',
    '',
    '── INPUT ──────────────────────────────────────────────────',
    f'Raw addresses file        : {RAW_ADDR_FILE}',
    f'Alias rules file          : {ALIAS_FILE}',
    f'Dictionary file           : {DICTIONARY_FILE}',
    f'Dim location file         : {DIM_LOCATION_FILE}',
    '',
    '── PART 1: NORMALIZATION & CITY FILTER ────────────────────',
    f'Total raw addresses loaded: {total_raw:,}',
    f'Total after normalization : {total_normalized:,}',
    f'Dropped (no city found)   : {total_dropped:,}',
    f'Passed to Part 2          : {total_passed_p2:,}',
    f'Filtered dim_location rows: {len(filtered_dim_df):,}',
    '',
    '── PART 2: FUZZY MATCHING ─────────────────────────────────',
    f'Fuzzy threshold           : {FUZZY_THRESHOLD}%',
    f'Total valid matches       : {total_valid:,}',
    f'  ↳ Level 1 (barangay)   : {level_counts[1]:,}',
    f'  ↳ Level 2 (city)       : {level_counts[2]:,}',
    f'  ↳ Level 3 (province)   : {level_counts[3]:,}',
    f'Total invalid matches     : {total_invalid:,}',
    f'Combined addresses file   : {combined_path}',
    f'Low-confidence threshold  : {low_confidence_threshold:.3f}',
    f'Low-confidence rows       : {len(low_confidence_out_df):,}',
    '',
    '── OUTPUT FILES ───────────────────────────────────────────',
    f'Valid addresses file      : {valid_path}',
    f'Combined addresses file   : {combined_path}',
    f'Invalid addresses file    : {invalid_path}',
    f'Low-confidence file       : {low_confidence_path}',
    f'Log file                  : {log_path}',
    f'Normalized addresses CSV  : normalized_addresses.csv',
    '=' * 60,
]

log_text = '\n'.join(log_lines)

with open(log_path, 'w', encoding='utf-8') as f:
    f.write(log_text)

print(log_text)
print(f'\n📋 Log saved → {log_path}')

   PHILIPPINE ADDRESS MAPPING PIPELINE — RUN LOG
Run Timestamp             : 2026-04-16 10:17:48

── INPUT ──────────────────────────────────────────────────
Raw addresses file        : ../../data/sample/ps_address_v3.csv
Alias rules file          : ../../data/utils/ph_address_alias_extended_v4.csv
Dictionary file           : ../../data/dictionary/ph_area_dictionary_v2.json
Dim location file         : ../../data/mapping/dim_location_20260415_v3.csv

── PART 1: NORMALIZATION & CITY FILTER ────────────────────
Total raw addresses loaded: 122,888
Total after normalization : 122,888
Dropped (no city found)   : 65,347
Passed to Part 2          : 57,541
Filtered dim_location rows: 36,348

── PART 2: FUZZY MATCHING ─────────────────────────────────
Fuzzy threshold           : 80%
Total valid matches       : 57,541
  ↳ Level 1 (barangay)   : 13,710
  ↳ Level 2 (city)       : 43,826
  ↳ Level 3 (province)   : 5
Total invalid matches     : 0
Combined addresses file   : ..\..\data\output\combined

In [ ]:
# Fast post-processing: add raw_deliveraddress to valid output
# Uses in-memory variables from the finished run (no fuzzy rerun)

bridge_df = pd.DataFrame({
    "raw_deliveraddress": raw_addresses,
    "order_deliveraddress": normalized_addresses
})

# Handle duplicates safely by matching occurrence order per normalized address
bridge_df["_ord"] = bridge_df.groupby("order_deliveraddress").cumcount()

valid_plus_df = valid_out_df.copy()
valid_plus_df["_ord"] = valid_plus_df.groupby("order_deliveraddress").cumcount()

valid_plus_df = valid_plus_df.merge(
    bridge_df[["order_deliveraddress", "_ord", "raw_deliveraddress"]],
    on=["order_deliveraddress", "_ord"],
    how="left"
).drop(columns="_ord")

# Put raw column first
cols = ["raw_deliveraddress"] + [c for c in valid_plus_df.columns if c != "raw_deliveraddress"]
valid_plus_df = valid_plus_df[cols]

# Save a new file (keeps original valid file untouched)
valid_with_raw_path = valid_path.with_name(f"{valid_path.stem}_with_raw{valid_path.suffix}")
valid_plus_df.to_excel(valid_with_raw_path, index=False)
print(f"Saved: {valid_with_raw_path} | rows={len(valid_plus_df):,}")

Saved: ..\..\data\output\valid\ps_valid_addresses_(part1)20260416_with_raw.xlsx | rows=57,541


In [ ]:
# Cell 14 — Remove addresses shorter than 18 characters (fast post-processing)
from pathlib import Path
import pandas as pd

# Prefer the freshly created output; fallback to a known filename
candidate_paths = [
    Path(valid_with_raw_path) if 'valid_with_raw_path' in globals() else None,
    Path('../../data/output/valid/valid_addresses_20260415_with_raw.xlsx'),
]
in_path = next((p for p in candidate_paths if p is not None and p.exists()), None)

if in_path is None:
    raise FileNotFoundError(
        'No input file found. Add valid_addresses_20260415_with_raw.xlsx under ../../data/output/valid/ '
        'or run the previous post-processing cell first.'
    )

out_path = in_path.with_name(f"{in_path.stem}_min18{in_path.suffix}")
df = pd.read_excel(in_path)

# Use raw address length when available; otherwise fallback to normalized column
source_col = 'raw_deliveraddress' if 'raw_deliveraddress' in df.columns else 'order_deliveraddress'

filtered_df = df[df[source_col].fillna('').astype(str).str.strip().str.len() >= 18].copy()
removed_df = df[df[source_col].fillna('').astype(str).str.strip().str.len() < 18].copy()

filtered_df.to_excel(out_path, index=False)

print(f'Input file     : {in_path}')
print(f'Length column  : {source_col}')
print(f'Input rows     : {len(df):,}')
print(f'Kept rows      : {len(filtered_df):,}')
print(f'Removed rows   : {len(removed_df):,}')
print(f'Saved file     : {out_path}')

if not removed_df.empty:
    print('\nSample removed addresses:')
    display(removed_df[[source_col]].head(10))

Input file     : ..\..\data\output\valid\ps_valid_addresses_(part1)20260416_with_raw.xlsx
Length column  : raw_deliveraddress
Input rows     : 57,541
Kept rows      : 41,638
Removed rows   : 15,903
Saved file     : ..\..\data\output\valid\ps_valid_addresses_(part1)20260416_with_raw_min18.xlsx

Sample removed addresses:


,raw_deliveraddress
0,Santo Domingo St
3,Cainta Rizal
7,Casa Amaya
8,tapaz capiz
13,sta ros alaguna
14,217 Roxas St.
16,32 PILAR ROAD
20,splash n dine
24,BIR Bulua CDO
29,PIAT STA BARBARA
